# Projeto 2 — Sprint 2

## Teste de hipótese da proporção

**Variável:** Tempo de carregamento (min)  
**Planilha 1:** observações antes da intervenção.  
**Planilha 2:** observações depois da intervenção.

Este notebook segue a organização e o estilo dos notebooks da Sprint 1. Coloque os dois arquivos Excel na sessão do Colab antes de executar as células.


## 1. Bibliotecas e carregamento dos dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from pathlib import Path

def localizar_arquivo(caminho_repositorio, nome_upload):
    opcoes = [
        Path(caminho_repositorio),
        Path("../") / caminho_repositorio,
        Path("../../") / caminho_repositorio,
        Path(nome_upload)
    ]
    for caminho in opcoes:
        if caminho.exists():
            return str(caminho)
    raise FileNotFoundError(
        f"Arquivo não encontrado: {nome_upload}. "
        "Execute o notebook a partir da pasta principal do repositório "
        "ou envie a planilha para a sessão do Colab."
    )

ARQUIVO_ANTES = localizar_arquivo(
    "dados/planilha-1-antes-intervencao.xlsx",
    "Planilha1_Tempo de carregamento de caminhões com grãos.xlsx"
)
ARQUIVO_DEPOIS = localizar_arquivo(
    "dados/planilha-2-depois-intervencao.xlsx",
    "Planilha2_Tempo de carregamento de caminhões com grãos.xlsx"
)
COLUNA = "Tempo de carregamento (min)"

df_antes = pd.read_excel(ARQUIVO_ANTES)
df_depois = pd.read_excel(ARQUIVO_DEPOIS)

antes = pd.to_numeric(df_antes[COLUNA], errors="coerce").dropna()
depois = pd.to_numeric(df_depois[COLUNA], errors="coerce").dropna()

print("Antes:", len(antes), "observações")
print("Depois:", len(depois), "observações")

## Questão 1 — Proporções antes e depois

Na Questão 5 da Sprint 1, o grupo escolheu o intervalo **[20, 43] minutos** para calcular a probabilidade de referência. Já na Questão 7.4(b), foi utilizado o intervalo **[20, 30] minutos**.

Como os intervalos são diferentes, os valores não podem ser comparados diretamente no mesmo teste. Para manter o intervalo da Questão 5, este notebook calcula novamente a proporção da Planilha 2 em [20, 43]. O valor de [20, 30] também será exibido para comparação com o resultado registrado na Sprint 1.

In [ ]:
a = 20
b = 43

media_antes = antes.mean()
dp_antes = antes.std(ddof=1)

# Probabilidade calculada na Questão 5 da Sprint 1.
p = stats.norm.cdf(b, loc=media_antes, scale=dp_antes) - stats.norm.cdf(a, loc=media_antes, scale=dp_antes)

# Proporção depois, recalculada no mesmo intervalo [20, 43].
quantidade_depois = int(depois.between(a, b, inclusive="both").sum())
phat = quantidade_depois / len(depois)

# Proporção registrada na Sprint 1 para o intervalo [20, 30].
quantidade_20_30 = int(depois.between(20, 30, inclusive="both").sum())
phat_20_30 = quantidade_20_30 / len(depois)

print(f"Probabilidade p no intervalo [{a}, {b}]: {p:.4%}")
print(f"Quantidade depois em [{a}, {b}]: {quantidade_depois}")
print(f"Proporção depois em [{a}, {b}]: {phat:.2%}")
print(f"Proporção depois em [20, 30]: {phat_20_30:.2%}")

## Questão 3a — Formulação das hipóteses

O objetivo é verificar se aumentou a proporção de carregamentos dentro do intervalo de referência. O teste é unilateral à direita:

- **H₀:** p = p₀
- **H₁:** p > p₀

Neste notebook, p₀ é a probabilidade da Questão 5 da Sprint 1, e p̂ é recalculado na Planilha 2 usando o mesmo intervalo [20, 43].

In [ ]:
n = len(depois)
z_calculado = (phat - p) / np.sqrt(p * (1 - p) / n)

print(f"p = {p:.4f}")
print(f"p chapéu = {phat:.4f}")
print(f"n = {n}")
print(f"Valor de z calculado: {z_calculado:.4f}")

## Questão 3b — Teste com nível de significância de 5%

In [ ]:
alpha = 0.05
z_critico_5 = stats.norm.ppf(1 - alpha)

print(f"Valor crítico: {z_critico_5:.4f}")
print(f"Região de rejeição: z >= {z_critico_5:.4f}")
print(f"Região de não rejeição: z < {z_critico_5:.4f}")

if z_calculado >= z_critico_5:
    print("Decisão: rejeitar H0.")
    print("Conclusão: há evidências de aumento da proporção no intervalo escolhido.")
else:
    print("Decisão: não rejeitar H0.")
    print("Conclusão: não há evidências suficientes de aumento da proporção no intervalo escolhido.")

In [ ]:
x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x)

plt.figure(figsize=(9, 5))
plt.plot(x, y)
plt.fill_between(x, 0, y, where=(x >= z_critico_5), alpha=0.3,
                 label="Região de rejeição")
plt.axvline(z_critico_5, color="red", linestyle="--",
            label=f"Valor crítico = {z_critico_5:.2f}")
plt.axvline(z_calculado, color="green", linestyle=":",
            label=f"z calculado = {z_calculado:.2f}")
plt.title("Teste de hipótese da proporção — 5%")
plt.xlabel("z")
plt.ylabel("Densidade")
plt.legend()
plt.show()

## Teste com nível de significância de 1%

In [ ]:
alpha = 0.01
z_critico_1 = stats.norm.ppf(1 - alpha)

print(f"Valor crítico: {z_critico_1:.4f}")
print(f"Região de rejeição: z >= {z_critico_1:.4f}")
print(f"Região de não rejeição: z < {z_critico_1:.4f}")

if z_calculado >= z_critico_1:
    print("Decisão: rejeitar H0.")
    print("Conclusão: há evidências de aumento da proporção no intervalo escolhido.")
else:
    print("Decisão: não rejeitar H0.")
    print("Conclusão: não há evidências suficientes de aumento da proporção no intervalo escolhido.")

In [ ]:
x = np.linspace(-4, 4, 1000)
y = stats.norm.pdf(x)

plt.figure(figsize=(9, 5))
plt.plot(x, y)
plt.fill_between(x, 0, y, where=(x >= z_critico_1), alpha=0.3,
                 label="Região de rejeição")
plt.axvline(z_critico_1, color="red", linestyle="--",
            label=f"Valor crítico = {z_critico_1:.2f}")
plt.axvline(z_calculado, color="green", linestyle=":",
            label=f"z calculado = {z_calculado:.2f}")
plt.title("Teste de hipótese da proporção — 1%")
plt.xlabel("z")
plt.ylabel("Densidade")
plt.legend()
plt.show()

## Questão 3c — Comparação com a análise preliminar

Compare as decisões dos testes com os resultados descritivos da Sprint 1.

**Atenção:** o relatório da Sprint 1 apresentou 57% na faixa [20, 30]. A probabilidade p da Questão 5 foi calculada para [20, 43]. Por isso, para realizar o teste com o mesmo evento, p̂ foi recalculado para [20, 43]. Se a correção do professor indicar outro intervalo, altere os limites `a` e `b` e execute novamente.

A aproximação normal utilizada no teste z deve ser interpretada com cuidado quando a proporção esperada de fracassos for muito pequena. Não rejeitar H₀ não significa provar que as proporções são iguais.